In [ ]:

# ============================================================
# PROJECT 1 - SMART CITY TRAFFIC ANALYTICS
# REAL UCI DATA | GOOGLE COLAB | SINGLE CELL
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "pandas", "numpy", "plotly", "ucimlrepo"
])

import pandas as pd
import numpy as np
import plotly.express as px
import zipfile
from pathlib import Path
from plotly.io import to_html
from IPython.display import display
from ucimlrepo import fetch_ucirepo

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

PROJECT = Path("/content/Smart_City_Traffic_Analytics")
PROJECT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ZIP = False

SOURCE_URL = (
    "https://archive.ics.uci.edu/dataset/492/"
    "metro+interstate+traffic+volume"
)

# ------------------------------------------------------------
# 2. DOWNLOAD REAL DATASET
# ------------------------------------------------------------

print("Loading real UCI traffic dataset...")

try:
    dataset = fetch_ucirepo(id=492)
    df = dataset.data.original.copy()
    print("Loaded using UCI Python package.")

except Exception as error:
    print("UCI API unavailable:", error)
    print("Trying official CSV download...")

    csv_url = (
        "https://archive.ics.uci.edu/static/public/492/"
        "Metro_Interstate_Traffic_Volume.csv.gz"
    )

    df = pd.read_csv(csv_url)
    print("Loaded using official UCI CSV.")

required = ["date_time", "traffic_volume"]

if not all(c in df.columns for c in required):
    raise ValueError(
        "Required traffic columns missing. "
        f"Available columns: {list(df.columns)}"
    )

print("\nDATASET LOADED SUCCESSFULLY")
print("Original records:", len(df))
print("Columns:", list(df.columns))

# ------------------------------------------------------------
# 3. DATA CLEANING
# ------------------------------------------------------------

original_rows = len(df)

df["date_time"] = pd.to_datetime(
    df["date_time"],
    errors="coerce"
)

df["traffic_volume"] = pd.to_numeric(
    df["traffic_volume"],
    errors="coerce"
)

df = df.dropna(
    subset=["date_time", "traffic_volume"]
)

df = df[df["traffic_volume"] >= 0]

df = df.drop_duplicates().reset_index(drop=True)

if df.empty:
    raise ValueError("No usable traffic records.")

df["Year"] = df["date_time"].dt.year
df["Month"] = df["date_time"].dt.month
df["Day"] = df["date_time"].dt.day_name()
df["Hour"] = df["date_time"].dt.hour

df["Is_Weekend"] = (
    df["date_time"].dt.dayofweek >= 5
)

if "temp" in df.columns:
    df["temp"] = pd.to_numeric(
        df["temp"], errors="coerce"
    )
    df["Temperature_C"] = df["temp"] - 273.15

df.to_csv(
    PROJECT / "cleaned_traffic_data.csv",
    index=False
)

print("\nDATA CLEANING COMPLETED")
print("Original records:", original_rows)
print("Clean records:", len(df))

display(df.head())

# ------------------------------------------------------------
# 4. KPI SUMMARY
# ------------------------------------------------------------

kpis = {
    "Total Records": len(df),
    "Average Traffic Volume": round(
        df["traffic_volume"].mean(), 2
    ),
    "Maximum Traffic Volume": df["traffic_volume"].max(),
    "Minimum Traffic Volume": df["traffic_volume"].min(),
    "Years Covered": df["Year"].nunique(),
    "Weekend Records": int(df["Is_Weekend"].sum())
}

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

print("\nPROJECT KPIs")
display(kpi_df)

# ------------------------------------------------------------
# 5. INTERACTIVE PLOTLY CHARTS
# ------------------------------------------------------------

figures = []

def show_chart(fig, title):
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(l=45, r=40, t=80, b=55)
    )
    figures.append((title, fig))
    fig.show()

# CHART 1 - HOURLY TRAFFIC

hourly = (
    df.groupby("Hour", as_index=False)["traffic_volume"]
    .mean()
)

show_chart(
    px.line(
        hourly,
        x="Hour",
        y="traffic_volume",
        markers=True
    ),
    "1. Average Traffic Volume by Hour"
)

# CHART 2 - DAILY TRAFFIC

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

daily = (
    df.groupby("Day", as_index=False)["traffic_volume"]
    .mean()
)

daily["Day"] = pd.Categorical(
    daily["Day"],
    categories=day_order,
    ordered=True
)

daily = daily.sort_values("Day")

show_chart(
    px.bar(
        daily,
        x="Day",
        y="traffic_volume",
        color="traffic_volume"
    ),
    "2. Average Traffic Volume by Day"
)

# CHART 3 - MONTHLY TRAFFIC

monthly = (
    df.groupby("Month", as_index=False)["traffic_volume"]
    .mean()
    .sort_values("Month")
)

show_chart(
    px.line(
        monthly,
        x="Month",
        y="traffic_volume",
        markers=True
    ),
    "3. Average Traffic Volume by Month"
)

# CHART 4 - YEARLY TRAFFIC

yearly = (
    df.groupby("Year", as_index=False)["traffic_volume"]
    .mean()
    .sort_values("Year")
)

show_chart(
    px.bar(
        yearly,
        x="Year",
        y="traffic_volume",
        color="traffic_volume"
    ),
    "4. Average Traffic Volume by Year"
)

# CHART 5 - WEEKDAY VS WEEKEND

weekend = (
    df.groupby("Is_Weekend", as_index=False)[
        "traffic_volume"
    ].mean()
)

weekend["Day_Type"] = weekend["Is_Weekend"].map({
    False: "Weekday",
    True: "Weekend"
})

show_chart(
    px.bar(
        weekend,
        x="Day_Type",
        y="traffic_volume",
        color="Day_Type"
    ),
    "5. Weekday vs Weekend Traffic"
)

# CHART 6 - WEATHER CONDITIONS

if "weather_main" in df.columns:

    weather = (
        df.groupby(
            "weather_main", as_index=False
        )["traffic_volume"].mean()
    )

    show_chart(
        px.bar(
            weather,
            x="weather_main",
            y="traffic_volume",
            color="traffic_volume"
        ),
        "6. Traffic Volume by Weather Condition"
    )

# CHART 7 - TEMPERATURE VS TRAFFIC

if "Temperature_C" in df.columns:

    scatter = df.dropna(
        subset=["Temperature_C", "traffic_volume"]
    )

    if len(scatter) > 10000:
        scatter = scatter.sample(
            10000, random_state=42
        )

    show_chart(
        px.scatter(
            scatter,
            x="Temperature_C",
            y="traffic_volume",
            opacity=0.4
        ),
        "7. Temperature vs Traffic Volume"
    )

# CHART 8 - DAY AND HOUR HEATMAP

heat = df.pivot_table(
    index="Day",
    columns="Hour",
    values="traffic_volume",
    aggfunc="mean"
)

heat = heat.reindex(day_order)

show_chart(
    px.imshow(
        heat,
        aspect="auto",
        labels={
            "x": "Hour",
            "y": "Day",
            "color": "Average Traffic"
        }
    ),
    "8. Traffic Pattern by Day and Hour"
)

# ------------------------------------------------------------
# 6. EXPORT CSV REPORTS
# ------------------------------------------------------------

for group in [
    "Year", "Month", "Day", "Hour", "Is_Weekend"
]:
    report = (
        df.groupby(group)["traffic_volume"]
        .agg(
            Average_Traffic="mean",
            Maximum_Traffic="max",
            Minimum_Traffic="min",
            Records="count"
        )
        .reset_index()
    )

    report.to_csv(
        PROJECT / f"summary_{group.lower()}.csv",
        index=False
    )

if "weather_main" in df.columns:
    report = (
        df.groupby("weather_main")["traffic_volume"]
        .agg(
            Average_Traffic="mean",
            Records="count"
        )
        .reset_index()
    )

    report.to_csv(
        PROJECT / "summary_weather.csv",
        index=False
    )

# ------------------------------------------------------------
# 7. COMBINED INTERACTIVE DASHBOARD
# ------------------------------------------------------------

html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Smart City Traffic Analytics</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p {
    text-align: center;
}
.chart {
    background: white;
    border-radius: 12px;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
}
</style>
</head>
<body>
<h1>SMART CITY TRAFFIC ANALYTICS</h1>
<p>Real Historical Interstate Traffic Data</p>
"""

for i, (title, fig) in enumerate(figures):
    html += (
        '<div class="chart">'
        + to_html(
            fig,
            full_html=False,
            include_plotlyjs=(i == 0)
        )
        + "</div>"
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Smart_City_Traffic_Dashboard.html"
)

dashboard_path.write_text(
    html,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. README
# ------------------------------------------------------------

readme = f"""
SMART CITY TRAFFIC ANALYTICS

Dataset:
{SOURCE_URL}

Dataset Provider:
UCI Machine Learning Repository

Original Dataset:
Metro Interstate Traffic Volume

Clean Records:
{len(df)}

Interactive Charts:
{len(figures)}

Analysis:
Hourly Traffic
Daily Traffic
Monthly Traffic
Yearly Traffic
Weekday vs Weekend
Weather Conditions
Temperature vs Traffic
Day-Hour Heatmap

No synthetic traffic records generated.

Dataset limitation:
Traffic from one westbound interstate
monitoring location, not an entire city.
"""

(PROJECT / "README.txt").write_text(
    readme,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 9. CREATE ZIP
# ------------------------------------------------------------

zip_path = Path(
    "/content/Smart_City_Traffic_Analytics.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:
    for file in PROJECT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )

# ------------------------------------------------------------
# 10. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 45)
print("PROJECT 1 COMPLETED")
print("=" * 45)
print("Clean records:", len(df))
print("Interactive charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
